# CNN tipo VGG en TensorFlow: batch size, tamaño de dataset y regularización L1/L2

- Objetivo: medir, no asumir, el efecto real de tres hiperparámetros sobre una CNN tipo VGG entrenada en CIFAR-10: tamaño de batch, tamaño del set de entrenamiento y regularización L1/L2.
- Este notebook acompaña al artículo completo, con el detalle de las decisiones de ingeniería, el diccionario de parámetros y la interpretación de cada gráfica: **artículo completo →** https://fuzzyfrog.ai/es/ai-lab/proyectos/educacion/cnn-vgg-batch-size-regularizacion-l1-l2-tensorflow-keras/
- **Dataset:** CIFAR-10, público y abierto (https://www.cs.toronto.edu/~kriz/cifar.html). No requiere ningún tratamiento de confidencialidad.
- **Nota de reproducibilidad:** el notebook original de clase repetía el mismo bloque de código para cada configuración de batch size, tamaño de dataset y regularización. Aquí se refactorizó en una sola función reutilizable, para que el experimento sea más fácil de extender y de auditar.


## Diccionario de parámetros (según TensorFlow/Keras)

Antes de tocar el código, esto es lo que significa cada parámetro que vas a ver repetido:

| Parámetro | Dónde aparece | Qué controla |
|---|---|---|
| `filters` | `Conv2D` | Cuántos mapas de características distintos aprende esa capa |
| `kernel_size` | `Conv2D` | El tamaño de la ventana que se desliza sobre la imagen, ej. `(3,3)` |
| `strides` | `Conv2D`, `MaxPooling2D` | Cuántos píxeles se desplaza la ventana en cada paso |
| `padding` | `Conv2D`, `MaxPooling2D` | `'same'` conserva el tamaño espacial de salida; `'valid'` lo reduce |
| `pool_size` | `MaxPooling2D` | El tamaño de la ventana que se reduce a un solo valor (el máximo) |
| `units` | `Dense` | Cuántas neuronas tiene esa capa completamente conectada |
| `activation` | Cualquier capa | La función no lineal que decide qué tan "activa" queda cada neurona |
| `kernel_regularizer` | `Dense`, `Conv2D` | Penaliza pesos grandes: `regularizers.L1(lambda)`, `L2(lambda)` o `L1L2(lambda)` |
| `batch_size` | `model.fit()` | Cuántos ejemplos se procesan antes de actualizar los pesos una vez |
| `epochs` | `model.fit()` | Cuántas veces el modelo recorre el set de entrenamiento completo |
| `learning_rate` | El optimizador | Qué tan grande es el paso que da el optimizador en cada actualización |

**Conteo de parámetros entrenables (la fórmula que usa `model.summary()`):**
- Capa `Conv2D`: `(kernel_alto × kernel_ancho × canales_entrada × filtros) + filtros`
- Capa `Dense`: `(unidades_entrada × unidades_salida) + unidades_salida`

Ninguno de estos totales cambia con `batch_size` ni con la cantidad de datos de entrenamiento: esos son parámetros del *proceso* de entrenamiento, no de la arquitectura.

## Diagrama de la arquitectura base

`Conv2D(64, 3x3, tanh)` → `MaxPooling2D(2x2)` → `Flatten` → `Dense(1024, tanh)` → `Dense(10, softmax)`.

Esta misma arquitectura, sin cambiar una sola capa, es la que se reutiliza en los tres experimentos. El diagrama interactivo completo, con el conteo de parámetros de cada capa, está en el artículo (sección "Diagrama de la solución").

In [ ]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)

gpus = tf.config.experimental.list_physical_devices('GPU')
print("GPUs disponibles:", len(gpus))


## Carga de datos

- CIFAR-10: 60,000 imágenes de 32×32px a color, 10 clases, 50,000 de entrenamiento y 10,000 de prueba.
- Para agilizar el entrenamiento en un ejercicio de clase, se trabaja con subconjuntos reducidos del set original.

In [ ]:
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import models, layers, optimizers, regularizers
from tensorflow.keras.datasets import cifar10
import numpy as np
import time
import matplotlib.pyplot as plt
import sklearn.model_selection as model_selection

((trainX, trainY), (testX, testY)) = cifar10.load_data()
print("trainX:", trainX.shape, "| testX:", testX.shape)
print("trainY:", trainY.shape, "| testY:", testY.shape)


## Explicación de los datos

- Cada imagen es un tensor `(32, 32, 3)`: alto, ancho y 3 canales de color (RGB).
- Las etiquetas se convierten a one-hot con `to_categorical` porque la capa de salida usa `softmax` sobre 10 clases.
- Los píxeles se normalizan a `[0, 1]` dividiendo entre 255, para que el optimizador trabaje con valores en una escala estable.

In [ ]:
for i in range(9):
    plt.subplot(330 + 1 + i)
    plt.imshow(trainX[i])
plt.suptitle("Muestra de imágenes CIFAR-10 (sin normalizar)")
plt.show()


## Preparación reutilizable de splits

Se define una función para generar splits de entrenamiento/validación/prueba de distintos tamaños, y otra para normalizar y codificar las etiquetas. Ambas se reutilizan en los tres experimentos.

In [ ]:
def preparar_split(n_train, n_val, semilla=42):
    """Genera un split de tamaño controlado a partir del set completo de CIFAR-10."""
    trainx, valx, trainy, valy = model_selection.train_test_split(
        trainX, trainY, train_size=n_train, test_size=n_val, random_state=semilla
    )
    x_tr, x_val, x_te = trainx / 255.0, valx / 255.0, testX / 255.0
    y_tr = to_categorical(trainy, num_classes=10)
    y_val = to_categorical(valy, num_classes=10)
    y_te = to_categorical(testY, num_classes=10)
    return x_tr, y_tr, x_val, y_val, x_te, y_te

# Split base para el modelo inicial y el barrido de batch size
x_train, y_train, x_validation, y_validation, x_test, y_test = preparar_split(n_train=40000, n_val=10000)
print("x_train:", x_train.shape, "| x_validation:", x_validation.shape)


## Modelado

### 6.1 Función de entrenamiento reutilizable

Un solo punto de construcción de la arquitectura, parametrizado por batch size, learning rate y regularización opcional de las capas densas. Reutilizarla evita que una diferencia de código entre configuraciones contamine la comparación.

In [ ]:
def construir_y_entrenar(x_tr, y_tr, x_val, y_val, n_batch=256, n_epochs=20, lrate=0.001,
                          l1=None, l2=None, verbose=0):
    reg = None
    if l1 is not None and l2 is not None:
        reg = regularizers.L1L2(l1=l1, l2=l2)
    elif l1 is not None:
        reg = regularizers.L1(l1)
    elif l2 is not None:
        reg = regularizers.L2(l2)

    model = models.Sequential()
    model.add(layers.Conv2D(filters=64, kernel_size=(3, 3), strides=(1, 1),
                             padding='same', activation='tanh'))
    model.add(layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2), padding='valid'))
    model.add(layers.Flatten())
    model.add(layers.Dense(1024, activation='tanh', kernel_regularizer=reg))
    model.add(layers.Dense(10, activation='softmax', kernel_regularizer=reg))

    opt = optimizers.RMSprop(learning_rate=lrate)
    model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])

    t1 = time.time()
    historial = model.fit(x_tr, y_tr, validation_data=(x_val, y_val),
                           batch_size=n_batch, epochs=n_epochs, verbose=verbose)
    t2 = time.time()
    tiempo = t2 - t1
    print(f"batch_size={n_batch} | tiempo de entrenamiento: {tiempo:.2f}s")

    return model, historial, tiempo


def graficar_accuracy_loss(historial, titulo_extra=""):
    n_epochs = len(historial.history["accuracy"])
    plt.figure()
    plt.plot(np.arange(0, n_epochs), historial.history["accuracy"], label="train_acc")
    plt.plot(np.arange(0, n_epochs), historial.history["val_accuracy"], label="val_acc")
    plt.title(f"Accuracy de entrenamiento y validación {titulo_extra}")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend()
    plt.show()

    plt.figure()
    plt.plot(np.arange(0, n_epochs), historial.history["loss"], label="train_loss")
    plt.plot(np.arange(0, n_epochs), historial.history["val_loss"], label="val_loss")
    plt.title(f"Loss de entrenamiento y validación {titulo_extra}")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()
    plt.show()


### 6.2 Modelo inicial

**Por qué importa esta gráfica:** compara accuracy/loss de entrenamiento contra validación. Si el entrenamiento sigue mejorando mientras la validación se estanca o empeora, es la señal más directa de sobreajuste: el modelo está memorizando, no generalizando.

In [ ]:
modelo_inicial, hist_inicial, tiempo_inicial = construir_y_entrenar(
    x_train, y_train, x_validation, y_validation, n_batch=256, n_epochs=20, verbose=2
)
graficar_accuracy_loss(hist_inicial, "(modelo inicial, batch_size=256)")
modelo_inicial.summary()


In [ ]:
# Conteo manual de parámetros, siguiendo la fórmula de la sección de diccionario:
# Conv2D: (kernel_alto x kernel_ancho x canales_entrada) x filtros + filtros
conv_params = (3 * 3 * 3) * 64 + 64
# Dense 1024: 16384 (16x16x64 tras el pooling) x 1024 + 1024
dense1_params = (16 * 16 * 64) * 1024 + 1024
# Dense 10: 1024 x 10 + 10
dense2_params = 1024 * 10 + 10

print("Conv2D:", conv_params)
print("Dense(1024):", dense1_params)
print("Dense(10):", dense2_params)
print("Total:", conv_params + dense1_params + dense2_params)


### 6.3 Barrido de tamaño de batch

**Por qué importa esta gráfica:** mide el trade-off directo entre velocidad de cómputo y comportamiento del gradiente. Un batch más grande implica menos actualizaciones por época (más rápido), pero cada actualización promedia el gradiente sobre más ejemplos, lo que cambia la dinámica de convergencia.

In [ ]:
batch_sizes = [32, 128, 512, 1024]
resultados_batch = {}
tiempos_batch = []

for bs in batch_sizes:
    modelo, hist, tiempo = construir_y_entrenar(
        x_train, y_train, x_validation, y_validation, n_batch=bs, n_epochs=20, verbose=0
    )
    resultados_batch[bs] = hist
    tiempos_batch.append(tiempo)


In [ ]:
# Tiempo de entrenamiento vs tamaño de batch
plt.plot(batch_sizes, tiempos_batch, "r-^")
plt.xticks(batch_sizes)
plt.title("Batch size vs tiempo de entrenamiento")
plt.xlabel("Batch size"); plt.ylabel("Tiempo de ejecución (s)")
for i, bs in enumerate(batch_sizes):
    plt.text(bs, tiempos_batch[i] + 2, f"{tiempos_batch[i]:.1f}")
plt.show()


In [ ]:
# Comparación de curvas de aprendizaje entre tamaños de batch
plt.figure(figsize=(15, 10))
colores = {"32": "r", "128": "g", "512": "m", "1024": "y"}

for i, metric in enumerate(["accuracy", "loss", "val_accuracy", "val_loss"]):
    plt.subplot(2, 2, i + 1)
    for bs in batch_sizes:
        h = resultados_batch[bs]
        n_epochs = len(h.history[metric])
        plt.plot(np.arange(0, n_epochs), h.history[metric], marker=".",
                 color=colores[str(bs)], label=f"batch_size={bs}")
    plt.title(metric)
    plt.xlabel("Epoch"); plt.legend()

plt.tight_layout()
plt.show()


**Lectura de esta comparación:** en accuracy y loss de entrenamiento, batches chicos (32-128) tienden a comportarse mejor. Pero en loss de validación ocurre lo contrario: los batches grandes muestran menor loss de validación. Ver la sección "Proceso de iteración" del artículo para la interpretación completa de este resultado contraintuitivo.

### 6.4 Barrido de tamaño del set de entrenamiento

**Por qué importa esta gráfica:** aísla el efecto de la cantidad de datos, independientemente de la arquitectura o del batch size (fijo en 128, el mejor balance del experimento anterior). Es la pregunta más práctica de cualquier proyecto real: ¿vale la pena recolectar/usar más datos, dado el costo de cómputo que implica?

In [ ]:
tamanios = [(800, 200), (10000, 2500), (20000, 5000), (30000, 7500), (40000, 10000)]
resultados_tamanio = {}
tiempos_tamanio = []

for n_train, n_val in tamanios:
    x_tr, y_tr, x_val, y_val, _, _ = preparar_split(n_train=n_train, n_val=n_val)
    modelo, hist, tiempo = construir_y_entrenar(
        x_tr, y_tr, x_val, y_val, n_batch=128, n_epochs=20, verbose=0
    )
    resultados_tamanio[n_train] = hist
    tiempos_tamanio.append(tiempo)


In [ ]:
tam_x = [t[0] for t in tamanios]
plt.plot(tam_x, tiempos_tamanio, "r-^")
plt.xticks(tam_x)
plt.title("Tamaño de x_train vs tiempo de entrenamiento")
plt.xlabel("Tamaño de x_train"); plt.ylabel("Tiempo de ejecución (s)")
for i, n in enumerate(tam_x):
    plt.text(n, tiempos_tamanio[i] + 1, f"{tiempos_tamanio[i]:.1f}")
plt.show()


In [ ]:
# Comparación de curvas de aprendizaje entre tamaños de dataset
plt.figure(figsize=(15, 10))
colores_tam = ["r", "g", "m", "y", "b"]

for i, metric in enumerate(["accuracy", "loss", "val_accuracy", "val_loss"]):
    plt.subplot(2, 2, i + 1)
    for j, n_train in enumerate(tam_x):
        h = resultados_tamanio[n_train]
        n_epochs = len(h.history[metric])
        plt.plot(np.arange(0, n_epochs), h.history[metric], marker=".",
                 color=colores_tam[j], label=f"n_train={n_train}")
    plt.title(metric)
    plt.xlabel("Epoch"); plt.legend()

plt.tight_layout()
plt.show()


**Lectura de esta comparación:** todos los modelos siguen sobreajustados (buen desempeño en entrenamiento, peor en validación), pero el desempeño de validación mejora consistentemente con más datos, y las curvas se vuelven más estables (menos oscilación) al aumentar el tamaño del set. El número de parámetros entrenables no cambia con el tamaño del dataset, solo cambia el tiempo de cómputo y la calidad del ajuste.

### 6.5 Regularización L1 / L2

**Por qué importa esta gráfica:** muestra que la regularización no es un interruptor de "mejor/peor", es una perilla continua. Un lambda demasiado alto penaliza tanto los pesos que el modelo no puede aprender nada útil; uno demasiado bajo es indistinguible de no tener regularización.

In [ ]:
configs_reg = [
    {"nombre": "L1=10, L1L2=10",     "l1": 10,     "l2": None},
    {"nombre": "L2=0.1, L1L2=0.1",   "l1": None,   "l2": 0.1},
    {"nombre": "L1=0.0001",          "l1": 0.0001, "l2": None},
    {"nombre": "L2=0.0001",          "l1": None,   "l2": 0.0001},
]

resultados_reg = {}
tiempos_reg = []

for cfg in configs_reg:
    modelo, hist, tiempo = construir_y_entrenar(
        x_train, y_train, x_validation, y_validation, n_batch=128, n_epochs=20,
        l1=cfg["l1"], l2=cfg["l2"], verbose=0
    )
    resultados_reg[cfg["nombre"]] = hist
    tiempos_reg.append(tiempo)
    graficar_accuracy_loss(hist, f"({cfg['nombre']})")


**Lectura esperada:** con lambda=10, la penalización es tan fuerte que domina la función de pérdida, el modelo apenas aprende y accuracy se mantiene cerca del azar. Con lambda=0.0001, el efecto es casi indistinguible del modelo sin regularizar. El valor útil está en algún punto intermedio, y encontrarlo requiere, otra vez, medir y no asumir.

## Evaluación

Comparación final de las tres dimensiones exploradas: velocidad de cómputo (tiempo), calidad del ajuste (accuracy/loss de validación) y estabilidad (oscilación de las curvas).

In [ ]:
resumen = []
for bs in batch_sizes:
    h = resultados_batch[bs]
    resumen.append({
        "experimento": "batch_size", "configuracion": bs,
        "val_accuracy_final": h.history["val_accuracy"][-1],
        "val_loss_final": h.history["val_loss"][-1],
    })
for n_train in tam_x:
    h = resultados_tamanio[n_train]
    resumen.append({
        "experimento": "tamanio_dataset", "configuracion": n_train,
        "val_accuracy_final": h.history["val_accuracy"][-1],
        "val_loss_final": h.history["val_loss"][-1],
    })
for cfg in configs_reg:
    h = resultados_reg[cfg["nombre"]]
    resumen.append({
        "experimento": "regularizacion", "configuracion": cfg["nombre"],
        "val_accuracy_final": h.history["val_accuracy"][-1],
        "val_loss_final": h.history["val_loss"][-1],
    })

import pandas as pd
df_resumen = pd.DataFrame(resumen)
df_resumen.round(4)


## Hallazgos principales

- El **modelo inicial está sobreajustado** desde la primera corrida: buen desempeño en entrenamiento, pobre en validación. Ese es el punto de partida de todo el ejercicio, no un error a corregir antes de empezar.
- **Batch size y tiempo de entrenamiento son inversamente proporcionales**, pero el efecto sobre la calidad del modelo no es unidireccional: batches chicos ganan en accuracy/loss de entrenamiento, batches grandes ganan en loss de validación. No hay un "mejor batch size" universal, depende de qué se está optimizando.
- **Más datos de entrenamiento mejoran el desempeño de validación y estabilizan las curvas**, a costa de mayor tiempo de cómputo. El número de parámetros entrenables no cambia ni con el batch size ni con la cantidad de datos, solo depende de la arquitectura.
- **La regularización L1/L2 es una perilla, no un interruptor**: valores extremos de lambda apagan el aprendizaje; valores muy pequeños no tienen efecto observable. El valor útil está en un rango intermedio que solo se encuentra experimentando.
- La disciplina real que se practica aquí no es "saber qué hiperparámetro usar", es aislar una variable a la vez y leer la dinámica completa del entrenamiento, no solo el número final.